In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("snmahsa/animal-image-dataset-cats-dogs-and-foxes")

print("Path to dataset files:", path)

100%|██████████| 388M/388M [00:05<00:00, 74.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/snmahsa/animal-image-dataset-cats-dogs-and-foxes/versions/1


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt

# Veri ön işleme
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resimleri yeniden boyutlandır
    transforms.ToTensor(),  # Tensor'a çevir
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalizasyon
])

# Veri setlerini yükle
data_dir = 'datasets'
dataset = ImageFolder(data_dir, transform=transform)

# Veriyi eğitim ve test olarak böl
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

# Veri yükleyicileri oluştur
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# CNN modelini tanımla
class AnimalClassifier(nn.Module):
    def __init__(self):
        super(AnimalClassifier, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, 3)  # 3 sınıf için
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(-1, 64 * 28 * 28)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# Model, kayıp fonksiyonu ve optimizasyon
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AnimalClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Eğitim fonksiyonu
def train_model(num_epochs):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}')

# Test fonksiyonu
def test_model():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Doğruluk: {100 * correct / total:.2f}%')

# Modeli eğit
train_model(num_epochs=10)

# Test et
test_model()

# Modeli kaydet
torch.save(model.state_dict(), 'animal_classifier.pth')

Found 165 images belonging to 3 classes.
Found 41 images belonging to 3 classes.
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 48s 2s/step - accuracy: 0.4528 - loss: -15.7108 - val_accuracy: 0.4878 - val_loss: -121.4155
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - accuracy: 0.4922 - loss: -220.8290 - val_accuracy: 0.4878 - val_loss: -751.4976
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.4892 - loss: -1218.0762 - val_accuracy: 0.4878 - val_loss: -2866.9778
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - accuracy: 0.4627 - loss: -4155.7412 - val_accuracy: 0.4878 - val_loss: -8434.1172
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 82s 981ms/step - accuracy: 0.4291 - loss: -12307.5400 - val_accuracy: 0.4878 - val_loss: -20850.4824
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - accuracy: 0.4980 - loss: -28603.0410 - val_accuracy: 0.4878 - val_loss: -45233.5156
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - accuracy: 0.5248 - loss: -54286.3203 - val_accuracy: 0.4878 - val_loss: -89

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

# Modeli yükle

# Test görüntüsünü yükle ve ölçekle
test_image = image.load_img('helo.jpeg', target_size=(128, 128))  # Test edilecek resmin yolu
test_image = np.expand_dims(test_image, axis=0) / 255.0

# Tahmin yap
prediction = model.predict(test_image)

# Sonucu yazdır
if prediction > 0.5:
    print("Bu bir kedi!")
else:
    print("Bu bir kedi değil!")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Bu bir kedi!
